In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import anndata

In [ ]:
adata = sc.read_h5ad("./pos-margaret-0.3.h5ad")
adata_all = sc.read_h5ad("../../hvae/outputs/Thymus_HierarVi.h5ad")
adata.obs['Genotype'] = adata_all.obs['Genotype'].loc[adata.obs_names]

# get denoised expression
adata_all = adata_all[adata.obs_names,adata.var_names]
adata.X = adata_all.obsm['RNA_Z1_denoised']
adata.obsm["protein_Z1_denoised"] = adata_all.obsm["protein_Z1_denoised"]
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
adata

In [ ]:
# adata_protein = sc.AnnData(adata.obsm["protein_Z1_denoised"])
# sc.pp.normalize_geometric(adata_protein)
# sc.pp.log1p(adata_protein)

In [ ]:
# lineage assignment of cells was made on the basis of their genotype
# (CD4+ T cell lineage for MHCI−/−, AND, and OT-II mice, 
#  CD8+ T cell lineage for MHCII−/−, F5, and OT-I mice, 
#             unassigned for wild-type (B6) mice ).
lineage_map = {
    'B6' : 'unassigned',
    'MHCI-/-' : 'CD4 lineage',
    'AND' : 'CD4 lineage',
    'OT-II' : 'CD4 lineage',
    'MHCII-/-' : 'CD8 lineage',
    'F5' : 'CD8 lineage',
    'OT-I' : 'CD8 lineage'
}
adata.obs['lineage'] = adata.obs['Genotype'].astype(str).replace(lineage_map)

In [ ]:
bps = pd.read_csv("./branch_probabilities.csv", index_col=0).fillna(0)
bps = bps.loc[adata.obs_names,:]
bps

In [ ]:
adata.obs = pd.concat([adata.obs, bps], axis=1)

In [ ]:
sc.pl.embedding(adata,'X_met_embedding',color = ["metric_pseudotime_v2", 'annotations', "Genotype"], ncols=1)

In [ ]:
genes_to_plot = ["Cd4","Cd8a","Cd8b1","Rag1","Rag2","Cxcr4","Trbc1","Ccr9","Cd24a","Cd5",
                 "Ccr4","Cd69","Gata3","Bcl2","Runx3","Zbtb7b","Il7r","Ccr7","S1pr1","Cd55","Klf2","Sell","H2-K1"]

genes_to_plot = [var for var in genes_to_plot if var in  adata.var_names]
# adata_sub = adata[(adata.obs['metric_pseudotime_v2'].sort_values()).index, genes_to_plot]

In [ ]:
sc.pl.embedding(adata, 'X_met_embedding', color=['Cd4', "lineage", 'Genotype'])

In [ ]:
adata.obs_names.is_unique

In [ ]:
adata_cd4 = adata[adata.obs['lineage'].isin(["CD4 lineage", "unassigned"]),:].copy()
adata_cd8 = adata[adata.obs['lineage'].isin(["CD8 lineage", "unassigned"]),:].copy() #, "unassigned"

adata_cd4 = adata_cd4[(adata_cd4.obs["Mature CD4"] > 0.5) ,:] #& (adata_cd4.obs["annotations"]!='Mature CD8')
adata_cd8 = adata_cd8[(adata_cd8.obs["Mature CD8"] > 0.5) ,:] #& (adata_cd8.obs["annotations"]!='Mature CD4')

adata_cd4 = adata_cd4[(adata_cd4.obs['metric_pseudotime_v2'].sort_values()).index, genes_to_plot].copy()
adata_cd8 = adata_cd8[(adata_cd8.obs['metric_pseudotime_v2'].sort_values()).index, genes_to_plot].copy()

In [ ]:
adata_cd = adata_cd4.concatenate(adata_cd8, batch_key = 'batch')
sc.pl.embedding(adata_cd, 'X_met_embedding', color=['annotations', "lineage", 'batch'])

In [ ]:
sc.pl.embedding(adata_cd8, 'X_met_embedding', color=['Mature CD8', "lineage", 'annotations',"metric_pseudotime_v2"])

In [ ]:
sc.pl.embedding(adata_cd4, 'X_met_embedding', color=['Mature CD4', "lineage", 'annotations', "metric_pseudotime_v2"])

In [ ]:
print(adata_cd4.obs['lineage'].value_counts())
adata_cd8.obs['lineage'].value_counts()

In [ ]:
proteins = {'ADT_CD8b(Ly-3)_A0230': 'CD8b(Ly-3)',
 'ADT_CD8a_A0002': 'CD8a',
 'ADT_CD24_A0212': 'CD24',
 'ADT_CD69_A0197': 'CD69',
 'ADT_CD127(IL-7Ra)_A0198': 'CD127',
 'ADT_CD54_A0074': 'CD5',
 'ADT_TCRbchain_A0120': 'TCRbchain',
 'ADT_CD55(DAF)_A0558': 'CD55',
 'ADT_CD62L_A0112': 'CD62L',
 'ADT_CD4_A0001': 'CD4'}

In [ ]:
adata_cd4.obsm['protein_Z1_denoised'] = adata_cd4.obsm['protein_Z1_denoised'][proteins.keys()].copy().rename(columns = proteins)
adata_cd8.obsm['protein_Z1_denoised'] = adata_cd8.obsm['protein_Z1_denoised'][proteins.keys()].copy().rename(columns = proteins)

In [ ]:
# Steps to generate pseudotime heatmaps:
# 1. extract data (genes x cells)
# 2. create small bins and average within bin (per row) with summarize_all
# 3. cbind CD4 and CD8
# 4. normalize per row (z score and winsorize to make heatmap colors visible)
# 5. separate CD4 and CD8
# 6. take rolling average over bins within lineage to smooth the rows
# 6. plot with constant column size so pseudotimes (early) match between lineages
from scipy.stats import zscore, mstats
from PyComplexHeatmap import *

cd4_df = pd.DataFrame(adata_cd4.X, index=adata_cd4.obs_names, columns=adata_cd4.var_names)
cd8_df = pd.DataFrame(adata_cd8.X, index=adata_cd8.obs_names, columns=adata_cd8.var_names)

cd4_cd8 = pd.concat([cd4_df, cd8_df])
cd4_cd8 = cd4_cd8.apply(zscore)
cd4_cd8 = cd4_cd8.apply(lambda x: mstats.winsorize(x,limits=[0.05,0.05]))

cd4_df = cd4_cd8.iloc[0:cd4_df.shape[0],:].copy()
cd8_df = cd4_cd8.iloc[cd4_df.shape[0]:,:].copy()

bins_cd4 = pd.cut(adata_cd4.obs["metric_pseudotime_v2"], bins=500)
bins_cd4.name = 'bins'

cd4_df = pd.concat([bins_cd4,adata_cd4.obs["metric_pseudotime_v2"], cd4_df], axis=1)
cd4_df = cd4_df.groupby('bins').mean()
cd4_df = cd4_df.dropna()
cd4_df = cd4_df.set_index('metric_pseudotime_v2')
cd4_df = cd4_df.rolling(20).mean()
cd4_df = cd4_df.dropna()


bins_cd8 = pd.cut(adata_cd8.obs["metric_pseudotime_v2"], bins=500)
bins_cd8.name = 'bins'

cd8_df = pd.concat([bins_cd8,adata_cd8.obs["metric_pseudotime_v2"], cd8_df], axis=1)
cd8_df = cd8_df.groupby('bins').mean()
cd8_df = cd8_df.dropna()
cd8_df = cd8_df.set_index('metric_pseudotime_v2')
cd8_df = cd8_df.rolling(20).mean()
cd8_df = cd8_df.dropna()




plot_data= pd.concat([cd4_df, cd8_df]).T
plot_data.columns = range(plot_data.shape[1])
top_annotation = pd.DataFrame(cd4_df.index.to_list()+cd8_df.index.to_list(), index = plot_data.columns, columns =  ['Pseudotime']) 
top_annotation = HeatmapAnnotation(Pseudotime=anno_simple(top_annotation.Pseudotime, cmap="autumn"))
col_split = pd.Series([1]*cd4_df.shape[0]+[2]*cd8_df.shape[0], index = plot_data.columns)

ClusterMapPlotter(plot_data, top_annotation=top_annotation, row_cluster=False,
        col_cluster=False,cmap='viridis',show_rownames=True, col_split=col_split, col_split_gap=1)



In [ ]:
# Steps to generate pseudotime heatmaps:
# 1. extract data (genes x cells)
# 2. create small bins and average within bin (per row) with summarize_all
# 3. cbind CD4 and CD8
# 4. normalize per row (z score and winsorize to make heatmap colors visible)
# 5. separate CD4 and CD8
# 6. take rolling average over bins within lineage to smooth the rows
# 6. plot with constant column size so pseudotimes (early) match between lineages
from scipy.stats import zscore, mstats
from PyComplexHeatmap import *
from sklearn.preprocessing import minmax_scale, robust_scale

cd4_df = adata_cd4.obsm['protein_Z1_denoised'].copy()
cd8_df = adata_cd8.obsm['protein_Z1_denoised'].copy()

cd4_cd8 = pd.concat([cd4_df, cd8_df])
cd4_cd8 = cd4_cd8.apply(zscore)
cd4_cd8 = cd4_cd8.apply(lambda x: mstats.winsorize(x,limits=[0.05,0.05]))

cd4_df = cd4_cd8.iloc[0:cd4_df.shape[0],:].copy()
cd8_df = cd4_cd8.iloc[cd4_df.shape[0]:,:].copy()

bins_cd4 = pd.cut(adata_cd4.obs["metric_pseudotime_v2"], bins=500)
bins_cd4.name = 'bins'

cd4_df = pd.concat([bins_cd4,adata_cd4.obs["metric_pseudotime_v2"], cd4_df], axis=1)
cd4_df = cd4_df.groupby('bins').mean()
cd4_df = cd4_df.dropna()
cd4_df = cd4_df.set_index('metric_pseudotime_v2')
cd4_df = cd4_df.rolling(20).mean()
cd4_df = cd4_df.dropna()


bins_cd8 = pd.cut(adata_cd8.obs["metric_pseudotime_v2"], bins=500)
bins_cd8.name = 'bins'

cd8_df = pd.concat([bins_cd8,adata_cd8.obs["metric_pseudotime_v2"], cd8_df], axis=1)
cd8_df = cd8_df.groupby('bins').mean()
cd8_df = cd8_df.dropna()
cd8_df = cd8_df.set_index('metric_pseudotime_v2')
cd8_df = cd8_df.rolling(20).mean()
cd8_df = cd8_df.dropna()




plot_data= pd.concat([cd4_df, cd8_df]).T
plot_data.columns = range(plot_data.shape[1])
top_annotation = pd.DataFrame(cd4_df.index.to_list()+cd8_df.index.to_list(), index = plot_data.columns, columns =  ['Pseudotime']) 
top_annotation = HeatmapAnnotation(Pseudotime=anno_simple(top_annotation.Pseudotime, cmap="autumn"))
col_split = pd.Series([1]*cd4_df.shape[0]+[2]*cd8_df.shape[0], index = plot_data.columns)

ClusterMapPlotter(plot_data, top_annotation=top_annotation, row_cluster=False,
        col_cluster=False,cmap='viridis',show_rownames=True, col_split=col_split, col_split_gap=1)



In [ ]:
# adata_cd4.obs = pd.concat([adata_cd4.obs,adata_cd4.obsm['protein_Z1_denoised']], axis=1)
# adata_cd8.obs = pd.concat([adata_cd8.obs,adata_cd8.obsm['protein_Z1_denoised']], axis=1)
adata_cd48_pro = adata_cd4.concatenate(adata_cd8)

In [ ]:
sc.pl.embedding(adata_cd48_pro, 'X_met_embedding', color=['CD8a', 'metric_pseudotime_v2'])
